# 03 · Feature extraction

For every hippocampal contact and trial:

| Feature | Key | Method |
|---|---|---|
| Local field potential | `lfp` | aligned, Laplacian-referenced signal |
| Theta power | `power` | complex Morlet wavelets at 3.5 Hz (3 cycles) and 7 Hz (6 cycles); 10·log10(amplitude²) |
| Theta phase clustering | `phaselock` | density of each trial's phase under a von Mises KDE (κ = 8) fitted across all trials, per frequency and time point |
| Theta phase speed | `speed` | instantaneous frequency (Hz): unwrapped phase, forward difference, × fs / 2π |

Theta features have shape (trials, 2, time) with index 0 = 3.5 Hz (slow theta) and 1 = 7 Hz (fast theta).
Wavelet transforms use the full epoch to avoid edge effects;
the stored arrays are cropped to −500…1000 ms around image onset. The computations are in `features.py`.

Input: `derivatives/epochs_hippocampus_aligned.pickle`. Output: `derivatives/features/Hippocampus_<participant>_<contact>.pkl`.

In [ ]:
import pickle

import numpy as np

import config
import features

config.FEATURES_DIR.mkdir(parents=True, exist_ok=True)
N_KEEP = int(round((config.ANALYSIS_TMAX - config.EPOCH_TMIN) * config.SFREQ))  # 768 samples: -500 to 1000 ms

In [ ]:
with open(config.ALIGNED_EPOCHS_FILE, "rb") as f:
    aligned_epochs = pickle.load(f)

for patient_id, epochs in aligned_epochs.items():
    sfreq = epochs.info["sfreq"]
    assert sfreq == config.SFREQ, f"{patient_id}: unexpected sampling rate {sfreq}"
    assert np.isclose(epochs.times[0], config.EPOCH_TMIN), f"{patient_id}: unexpected epoch start"
    memorability = epochs.metadata["vit_memorability"].to_numpy()

    for contact in epochs.ch_names:
        print(f"{patient_id} {contact}")
        data = epochs.get_data(picks=[contact])  # (trials, 1, times)
        assert data.shape[-1] > N_KEEP, "epoch shorter than the stored window"

        tfr = features.theta_tfr(data, sfreq)    # (trials, 2, times), complex
        contact_features = {
            "lfp": data[..., :N_KEEP],
            **features.theta_features(tfr, sfreq, n_keep=N_KEEP),
            "memorability": memorability,
        }

        with open(config.FEATURES_DIR / f"Hippocampus_{patient_id}_{contact}.pkl", "wb") as f:
            pickle.dump(contact_features, f)